# From reasoning to acting: tool use and code feedback


> The first two lectures treated a static model: lecture 02 used repeated sampling and voting at inference time, and lecture 03 let a model check another model's output. None of those methods lets the model interact with an environment. What the model knows is whatever was in its training data.
>
> This lecture closes a loop between the model and the environment: emit an action, run a tool, read back an observation. We split feedback by source into three layers: environment feedback (ReAct), execution feedback (RLEF), and AI feedback (Constitutional AI).

We start with a minimal example. Ask the model which year *Modern Literature* was founded, when that magazine is not in its training data.

First, the model answers from memory only. Following the impression that old literary magazines were founded early, it invents a year — say 1918. That year is made up, and it is wrong.

Second, the model is allowed to call a **search tool**. It emits an action Search[Modern Literature]. An outer program actually looks it up, and the tool returns "founded in Shanghai in 1923".

Third, the model reads that observation and answers 1923 — this time correctly.

The only difference is that in the second case the model does not merely talk: it looks something up, then answers from the retrieved fact. The model itself only emits text and cannot call an external program. The method is to wrap it in a program: the model writes text naming the tool it wants, the wrapper parses that text and actually runs the tool, the result is fed back as text, and the model decides the next step. Model and environment then form a **closed loop**: emit an action → run a tool → read back an observation → emit again.

By the end of this lecture we will have implemented that minimal loop. Placed in the minimal loop of lecture 1, action and feedback then had only the simplest form — the action was an arithmetic call, the feedback a numeric result. This lecture develops those two steps in full, in three layers by source of feedback: first the Agent receives environment feedback (ReAct), then feedback enters training (RLEF), and finally AI itself becomes a provider of feedback (Constitutional AI).

## 1. From reasoning to acting: the ReAct loop

This section addresses whether a model that only thinks, and never acts, is enough. The minimal loop in lecture 1 already let the model emit actions and call tools. Here we first look at what a think-only approach misses, then introduce the ReAct loop that alternates thinking and acting, and finally print a trajectory to meet its three kinds of fragment.

Consider the think-only approach. Before giving an answer, the model often writes a stretch of its own reasoning and emits reasoning and answer together. That "think then answer" method is chain-of-thought, abbreviated CoT. CoT lets the model think longer, but it uses only facts the model remembers and never touches the outside world. When the question needs knowledge the model does not have, it cannot produce a true answer, so it follows intuition and invents one that looks plausible. The longer the reasoning chain, the more chances to invent, and errors can also propagate down the chain.

ReAct takes a different order: do not finish thinking before acting; think one step, then act one step. The model first emits a Thought, stating what it intends to do; then an Action, naming the tool to call; the result returned by the tool (Observation) becomes the basis of the next round of thinking. The three fragments alternate. That is the split in the name ReAct: Reasoning and Acting each play a role.

The Observation in a trajectory must come from a real tool execution. It is inserted text, not a continuation the model wrote. On 134 ALFWorld games, the mean of ReAct's best 6 runs reaches 71%, while action-only Act is 45% — thinking keeps acting from losing its way. Below we print a paper-style trajectory and meet the three kinds of fragment.

A concrete example shows the boundary of CoT. Suppose we ask the model which year *Modern Literature* was founded, and the training data has no exact information about that magazine. CoT first writes a stretch of reasoning, then gives an answer. The reasoning touches no external source, so the model can only search internal memory. When it finds nothing, it usually does not answer "unknown"; it follows an intuition such as "literary magazines were founded early" and invents a year. The longer the chain, the more chances to invent.

The same question under ReAct does not depend on memory alone. The model first writes its current thought, then emits a search action, and the observation returned by the tool becomes the factual source of the next reasoning step. Thought grounds action; the observation from action revises the next thought. Below we print a ReAct trajectory by role and read it line by line.

In [ ]:
# A ReAct trajectory printed by role, to meet the three kinds of fragment in the loop
trajectory = [
    ("Thought", "I need to look up this magazine's founding year first."),
    ("Action", "Search[Modern Literature]"),
    ("Observation", "Modern Literature was a literary magazine founded in Shanghai in 1923."),
    ("Thought", "The founding year is 1923, later than 1919."),
    ("Action", "Finish[after]"),
]

for kind, content in trajectory:
    print(f"{kind:12s}{content}")
print()
print("Key observation: Action calls a tool; Observation comes from tool execution;")
print("Thought only updates the context and does not produce an observation.")


The printed trajectory has five records and covers ReAct's three kinds of fragment. We read them one by one.

The first is a Thought fragment: I need to look up this magazine's founding year first. It appears before the action, and its role is to write that subgoal into the context. Thought does not call a tool and does not produce an environment observation; it only updates the history the model itself sees.

The second is an Action fragment: Search[Modern Literature]. Action describes which tool to call and with what arguments. The tool name is Search, the argument is "Modern Literature". The model only decides the action; it does not execute it — the action is handed to a parser and a tool.

The third is an Observation fragment: Modern Literature was a literary magazine founded in Shanghai in 1923. It comes from a real tool execution. It is external text inserted into the context, not a continuation the model wrote. That is the boundary between ReAct and CoT: every fact in Observation must have the environment as its source.

The fourth Thought fragment reads the number 1923 from the previous observation, compares it with the 1919 given in the question, and concludes "after". The fifth Action fragment Finish[after] is a terminating action whose argument is the final answer; the loop stops when it sees that.

Stringing the five records together, the shape of the data flow is fixed: Thought writes a plan → Action calls a tool → Observation returns a result → Thought again → Action again. Thought grounds action; the observation from action revises the next thought. The two abilities support each other.

## 2. Defining and calling tools

This section explains what a tool is, and how the model represents a call. For the ReAct loop to run, there must be tools to call, and a way for the model to say which tool it wants. We first distinguish two kinds of content in the loop, then look at the three Wikipedia actions designed in the paper, and finally write a mini encyclopedia so the loop can run locally without a network.

Distinguish two kinds of content in the loop. In each round of model output, one kind is what the model itself is thinking, which we call Thought. It does not change the environment; it only updates the context the model sees, written $c_{t+1} = (c_t, \hat{a}_t)$. The other kind is a tool call that will actually run, written $a_t$. It executes in the environment and produces observation $o_{t+1}$. ReAct's action space is the union of the two $\hat{A} = A \cup L$, so "write a thought" is also treated as part of the action.

The paper designed three Wikipedia actions, deliberately weaker than a real retriever, to mimic how a person searches:
- `search[entity]`: returns the first few sentences of the entity page; if missing, some similar entities.
- `lookup[string]`: returns the next sentence on the page that contains that string, like Ctrl+F in a browser.
- `finish[answer]`: ends the task and gives an answer.

The three actions cover "find sources, inspect details, wrap up". The mini encyclopedia below uses the same interface, with a few local entries built in, so the loop can run in an environment without a network.

In [ ]:
class MiniWiki:
    """A mini encyclopedia: a few local entries, mimicking the paper's weakened retrieval interface.

    Corresponding paper actions: search(entity) returns the start of a page,
    lookup(keyword) returns the next sentence on the current page that contains
    the keyword. lookup uses a cursor to record the search position.
    """

    def __init__(self):
        self.pages = {
            "Modern Literature": [
                "Modern Literature was a literary magazine founded in Shanghai in 1923.",
                "Writers such as Lu Xun and Mao Dun published work in the magazine.",
                "The magazine continued into the early 1930s.",
            ],
            "May Fourth Movement": [
                "The May Fourth Movement began on 4 May 1919.",
                "It started with a student demonstration in Beijing and then spread nationwide.",
                "It is regarded as one of the starting points of modern Chinese history.",
            ],
        }
        self.current_page = []
        self.pos = 0

    def search(self, entity):
        """Return the first two sentences of the entity page; if missing, similar entity names."""
        self.current_page = self.pages.get(entity, [])
        self.pos = 0
        if self.current_page:
            return " ".join(self.current_page[:2])
        similar = [name for name in self.pages if entity in name]
        return f"Could not find \"{entity}\"; similar entities: {similar if similar else 'none'}"

    def lookup(self, keyword):
        """From the cursor, return the first sentence containing the keyword, or a notice if none."""
        for i in range(self.pos, len(self.current_page)):
            if keyword in self.current_page[i]:
                self.pos = i + 1
                return self.current_page[i]
        self.pos = len(self.current_page)
        return "No sentence containing that keyword"


wiki = MiniWiki()
print(wiki.search("Modern Literature"))
print(wiki.lookup("founded"))
print(wiki.search("a nonexistent entity"))


MiniWiki uses the same interface to mimic the paper's weakened retriever. The two methods correspond to two paper actions. We match the three calls against the output one by one.

First call `wiki.search("Modern Literature")`. search looks up the table: pages.get("Modern Literature", []) hits a built-in entry, sets current_page to that entry's list of three sentences, resets pos to 0, and returns the first two sentences joined. The first printed line is that page's summary, like the opening of a search-result page.

Second call `wiki.lookup("founded")`. lookup does not search the whole collection; it searches inside the page current_page points to. That is Ctrl+F in a browser: open the page first, then find a keyword on the page. lookup scans sentence by sentence from cursor pos. The first sentence contains "founded", so that sentence is returned and pos advances to 1. The cursor's point is that consecutive lookup calls do not keep returning the same sentence; they go deeper, as if a person were scrolling the page.

Third call `wiki.search("a nonexistent entity")`. pages.get returns an empty list and current_page is cleared. When search finds no entry it does not raise; it collects page names that contain the entity as similar-entity hints. Here there are none, so it returns 'Could not find "a nonexistent entity"; similar entities: none'.

Note the state dependence of search and lookup: lookup depends on search having set current_page first. If lookup is called first, current_page is still empty and nothing is found. Tools have to be used together. That is the source of the "find the page, then inspect details" order in the Agent loop.

**The action parser**

The model's output is free text and must be parsed into a structured action before it can run. The parser is lenient: it accepts both the paper format `Action: Search[entity]` and a function-call format `Action: search("entity")`. A single reply may contain several actions in a row; we keep them in order and execute them one by one. There are two termination signals: a `Finish[answer]` action, or a standalone `Final Answer: ...` line.

In [ ]:
import re


def parse_actions(text):
    """Take every action instruction from a model reply, in order.

    Supports two formats: Search[entity] and search("entity").
    Returns [(tool name, argument), ...], with tool names lowercased.
    """
    pattern = (
        r"Action\s*\d*\s*[:：]\s*([A-Za-z]+)"
        r"\s*(?:\[([^\]]*)\]|\(\s*(?:\"([^\"]*)\"|'([^']*)'|([^)]*))\s*\))"
    )
    actions = []
    for m in re.finditer(pattern, text):
        name = m.group(1).lower()
        arg = (m.group(2) or m.group(3) or m.group(4) or m.group(5) or "").strip()
        actions.append((name, arg))
    return actions


def extract_final_answer(text):
    """Extract the final answer from a reply; return None if none.

    Recognizes a Final Answer marker and a Finish[answer] action.
    """
    m = re.search(r"(?:Final Answer|final answer)\s*[:：]\s*([^\n]+)", text)
    if m:
        return m.group(1).strip()
    m = re.search(r"Action\s*\d*\s*[:：]\s*Finish\s*\[([^\]]*)\]", text)
    if m:
        return m.group(1).strip()
    return None


paper_reply = (
    "Thought: I need to look up the founding year first.\n"
    "Action 1: Search[Modern Literature]\n"
    "Thought: I already have the year.\n"
    "Action 2: Finish[after]"
)
print("Parsed actions:", parse_actions(paper_reply))
print("Final answer:", extract_final_answer(paper_reply))

scripted_reply = (
    'Thought: scripted reasoning: search first, then summarize.\n'
    'Action: search("CS329A self-improving agents")\n'
    'Final Answer: live API demo returns a placeholder conclusion.'
)
print("Parsed actions:", parse_actions(scripted_reply))
print("Final answer:", extract_final_answer(scripted_reply))

assert parse_actions(paper_reply) == [("search", "Modern Literature"), ("finish", "after")]
assert extract_final_answer(paper_reply) == "after"
assert parse_actions(scripted_reply) == [("search", "CS329A self-improving agents")]
assert extract_final_answer(scripted_reply) == "live API demo returns a placeholder conclusion."
print("Both formats are digested by the same parser; assertions passed.")


The parser's job is to turn a stretch of free text into a list of actions a program can run. We first look at what the input looks like, then at how the regex splits it.

Model output is not in one format. The paper format is `Action: Search[Modern Literature]`; some implementations let the model write a function call `Action: search("Modern Literature")`; the model may also write a numbered form `Action 1: Search[Modern Literature]`. The three writings mean the same thing, and the parser must read all of them. It returns a list [(tool name, argument), ...], with tool names lowercased so they match keys in the TOOLS dictionary.

The regex has three parts. The first, `Action\s*\d*\s*[:：]`, matches a fixed prefix: the word Action, optional spaces and a number (`\d*` matches "1"), and a colon (`[:：]` accepts both an ASCII colon and a fullwidth colon). The second, `([A-Za-z]+)`, captures the tool name, letters only, so Search, search, and Finish all match. The third matches the argument, in two writings: inside square brackets, any character except `]`, so `Search[Modern Literature]` yields "Modern Literature"; inside parentheses, a quoted string or bare text, so `search("CS329A self-improving agents")` yields the whole quoted span.

Hand-calculate one concrete reply. `Action 1: Search[Modern Literature]`: the first part matches `Action 1: `, the second captures Search, the third captures "Modern Literature" from the brackets, giving action ("search", "Modern Literature"). `re.finditer` finds every match in order, so several actions in one reply are returned in writing order and the loop executes them one by one. In the code, m.group(2) or m.group(3) or m.group(4) or m.group(5) takes the first nonempty argument group: bracket form only fills group(2), quoted form only fills group(3) or group(4), and a final or "" covers an empty argument.

**Assembling the loop**

The loop assembles tools, the parser, and the model. It maintains a message history and each step does four things: call the LLM for a reply; parse out actions; run tools; append Observation as a user message so the model can see it next time. There are two stopping conditions: a final answer appears, or the step count reaches the cap max_steps. The cap prevents a dead loop — a failure mode the paper names is emitting the same action over and over.


In [ ]:
import sys
import os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()

wiki = MiniWiki()
TOOLS = {"search": wiki.search, "lookup": wiki.lookup}


def run_react(client, question, instructions, tools, max_steps=8):
    """Run a full ReAct loop; return (final answer, trajectory list).

    client: llm_client client; question: the user question;
    instructions: format instructions for the model; tools: name-to-function map.
    The loop stops when a final answer appears, or when max_steps is exceeded.
    """
    messages = [{"role": "user",
                 "content": instructions + "\n\nQuestion: " + question}]
    trace = []
    pending = []       # remaining actions from one reply, still to execute
    finish = None

    for step in range(max_steps):
        if not pending:
            reply = client.chat(messages)
            trace.append("[model reply]\n" + reply)
            pending = parse_actions(reply)
            finish = extract_final_answer(reply)
            if not pending and finish is not None:
                trace.append("[end] " + finish)
                return finish, trace
            if not pending:
                messages.append({"role": "user",
                                 "content": "No action was recognized. Please give an Action or a Final Answer."})
                continue

        name, arg = pending.pop(0)
        if name == "finish":
            finish = arg
            break
        if name in tools:
            observation = tools[name](arg)
        else:
            observation = "Unknown tool: " + name
        trace.append(f"[run {name}({arg})] -> {observation}")
        messages.append({"role": "user", "content": "Observation: " + observation})

        if not pending:
            if finish is not None:
                trace.append("[end] " + finish)
                return finish, trace
            messages.append({"role": "user",
                             "content": "Please continue: give the next Action or Final Answer."})

    if finish is None:
        finish = "No answer within the step limit"
    trace.append("[end] " + finish)
    return finish, trace


print("run_react is defined: loop structure = parse action -> run tool -> inject observation -> stop.")


In [ ]:
question = "Was Modern Literature founded before or after the May Fourth Movement (1919)?"
instructions = (
    "Your knowledge base contains only a few local entries. You must look them up with the search tool before answering.\n"
    "Act step by step in the following format, writing one action per step:\n"
    "Thought: your reasoning\n"
    "Action: Search[entity] or Action: Lookup[keyword] or Action: Finish[answer]\n"
    "After Observation returns, keep thinking. When you have an answer, finish with Finish."
)

answer, trace = run_react(client, question, instructions, TOOLS, max_steps=6)
print("Question:", question)
print()
print("\n\n".join(trace))
print()
print("Final answer:", answer)
if False:
    print()
    print("Live API demo output is a placeholder: retrieval content and the final answer are generated by a script.")
    print("Under a live API the model would retrieve local entries and give a real comparison.")


Unfold a run of run_react step by step, and look at what the message history messages becomes at each step. Take the question Was Modern Literature founded before or after the May Fourth Movement (1919)? as an example. Below is a typical live run (the live API demo path gives a scripted placeholder trajectory; the data-flow shape is the same).

At initialization, messages has only one user message, whose content is instructions plus the question. That is the first text the model sees.

In round one, the model calls client.chat(messages) and emits a reply, usually starting with Thought and ending with Action:

Thought: I need to look up this magazine's founding year.
Action: Search[Modern Literature]

parse_actions takes one action ("search", "Modern Literature") from the reply and puts it in the pending queue. The loop pops that action from pending, finds that search maps to wiki.search in TOOLS, calls wiki.search("Modern Literature"), and gets the observation:

Modern Literature was a literary magazine founded in Shanghai in 1923. Writers such as Lu Xun and Mao Dun published work in the magazine.

The observation is wrapped as a user message and appended to messages, followed by a "please continue" hint. At this point messages is:

1. user: instructions + question
2. user: Observation: Modern Literature was a literary magazine founded in Shanghai in 1923. Writers such as Lu Xun and Mao Dun published work in the magazine.
3. user: Please continue: give the next Action or Final Answer.

Note that the model's own reply (Thought and Action) is not stored in messages. What is stored is the tool's execution result. This round the model is in effect saying "I will look up this magazine"; next round it sees the fact the tool retrieved for it.

In round two, the model sees the three messages above and emits a closing reply:

Thought: The founding year is 1923, later than 1919.
Action: Finish[after]

parse_actions yields ("finish", "after"). The loop recognizes the finish action and returns the argument "after" as the final answer. Two rounds end.

Stringing the two rounds together, the data flow is a closed chain: the model reads history → writes an action → the parser takes the action → the tool runs → the observation enters history → the model reads again. Every link of the loop is moving text: the model emits text, the parser extracts an action from text, the tool turns the action into new text, and the new text returns to the model's view. The everyday meaning of feedback is "send the result of the output back to the input". The ReAct loop is that kind of feedback loop.

**Silent thought versus retrieval**

On the same question, if we give only a CoT prompt and do not allow tool calls, the model can answer only from internal memory; when knowledge is missing it may invent a plausible answer. Below we script a CoT trajectory that invents a fact, then show a ReAct retrieval trajectory with a real observation from the local encyclopedia. A live model's behavior need not match the script — especially on a live API demo — but the structural difference is definite: ReAct at least emits one retrieval action and grounds the answer in an external observation.


In [ ]:
cot_hallucination = (
    "Thought: I have no exact memory of Modern Literature's founding year.\n"
    "Thought: Inferring from typical literary magazines, it may have been founded before 1919.\n"
    "Answer: it was founded before 1919."
)
print("Silent thought (CoT, no tools):")
print(cot_hallucination)
print()

# Ideal trajectory: a real observation from the local encyclopedia, showing how retrieval supplies facts
wiki_demo = MiniWiki()
ideal_react = [
    ("Thought", "I need to look up this magazine's founding year first."),
    ("Action", "Search[Modern Literature]"),
    ("Observation", wiki_demo.search("Modern Literature")),
    ("Thought", "The founding year is 1923, later than 1919."),
    ("Action", "Finish[after]"),
]
print("Ideal ReAct trajectory (observation from a real local-encyclopedia execution):")
for kind, content in ideal_react:
    print(f"{kind:12s}{content}")
print()
print("ReAct trajectory actually run (scripted placeholder):")
print("\n".join(trace))
print()
print("Contrast: ReAct grounds the answer in an external observation; CoT can only rely on internal memory.")
print("Among 50 HotpotQA trajectories the paper annotated by hand, 56% of CoT failures came from hallucinated reasoning;")
print("for ReAct that share is 0%, but invalid-retrieval errors rose by 23% — the two need to be combined.")


## 3. Code execution as a feedback signal

This section addresses how we know whether code the model wrote is correct. The search tool in the previous section still returns text, which may contain noise and may be misread by the model. Code execution gives another kind of feedback: actually run the code the model wrote. The program either passes the tests or yields a concrete error. The result is decided by the interpreter, not generated by the model, so this feedback is grounded — it comes from a real program run, not from the model's imagination.

Letting the model judge its own code, it often feels fine; switching the judge to the interpreter, the result is unique: pass is pass, error is error. Tests play that automatic criterion: a piece of code that runs all tests and passes them all is counted correct; any failure is counted wrong. The pass or error obtained by running the code is execution feedback. It is fed back to the model as an observation so the model knows how to change the next step.

Below we demonstrate that chain with a buggy function: run tests, collect results, and format failures into feedback text following the template in Appendix C of the paper.

In [ ]:
def run_tests(fn, tests):
    """Run a function and return per-test results.

    fn: function under test; tests: [(input, expected), ...]; a tuple input is unpacked as multiple arguments.
    """
    results = []
    for inputs, expected in tests:
        try:
            if isinstance(inputs, tuple):
                got = fn(*inputs)
            else:
                got = fn(inputs)
            results.append((inputs, expected, got, got == expected))
        except Exception as exc:
            results.append((inputs, expected, type(exc).__name__, False))
    return results


def format_feedback(results):
    """Format failed tests into feedback text for the model, following the template."""
    failed = [r for r in results if not r[3]]
    if not failed:
        return "All tests passed."
    lines = ["Your code failed the following tests:"]
    for inputs, expected, got, _ in failed:
        lines.append(f"- input {inputs} failed: Expected '{expected}' but got '{got}'")
    lines.append("Give it another try.")
    return "\n".join(lines)


def buggy_is_palindrome(s):
    """Palindrome check that skips comparing the first character (deliberately wrong)."""
    return s == s[1:]


pal_tests = [("racecar", True), ("hello", False), ("abba", True), ("a", True)]
results = run_tests(buggy_is_palindrome, pal_tests)
for inputs, expected, got, ok in results:
    print(f"input={inputs:8s} expected={expected} got={str(got):8s} ok={ok}")
print()
print(format_feedback(results))


run_tests runs each test once; format_feedback concatenates the failures into a stretch of feedback text for the model. We first hand-calculate each of the four tests and see where buggy_is_palindrome goes wrong.

The implementation of buggy_is_palindrome(s) is s == s[1:], which drops the first character from the comparison. It is actually comparing "the whole string" with "the string without the first character", and returns True only on inputs where those two happen to be equal.

- Input racecar: s[1:] is acecar, not equal to racecar, returns False. Expected True, the test fails.
- Input hello: s[1:] is ello, not equal to hello, returns False. Expected was already False, the test passes.
- Input abba: s[1:] is bba, not equal, returns False. Expected True, fails.
- Input a: s[1:] is the empty string, not equal, returns False. Expected True, fails.

Three of four fail. format_feedback concatenates those three by the template: a fixed first line, then one line per failure (input, expected, actual), then an encouraging last line. That text is fed back to the model as Observation, so the model knows which inputs were judged wrong and what the expected and actual values were.

The concrete format of the feedback template comes from Appendix C of the RLEF paper: list each failed test with expected and actual output, rather than a vague "the code is wrong". run_tests also catches exceptions and records the exception type, so the feedback includes what error was raised and the model can edit from that.

Tests are split into public and hidden. Public tests run at both train and inference time, and their results are fed to the model. Hidden tests run only at final scoring; the model never sees them. The reason for the split is direct: if the model could see every test, it could reverse-engineer answers from the test inputs instead of learning to write a correct function. Public tests guide the process; hidden tests judge the result.

**Feedback relevance**

Feeding execution feedback to the model is not the same as the model actually reading it. After a base model receives an error, it often emits the same wrong code again, as if it had not looked at the feedback. The RLEF paper (detailed in the next section) reports a counter-intuitive result: under a fixed sampling budget, trying several independent candidates is often stronger than iterative repair, for exactly this reason.

A random-feedback ablation in the paper confirms this further: replacing the feedback with an unrelated execution result from another problem clearly hurts repair. Feedback must be relevant to the error. Having feedback is not enough; the model still has to change the code according to it.

Below we reproduce that phenomenon with a scripted contrast: one side tries diverse independent candidates inside a fixed budget; the other keeps resubmitting the same wrong code.

In [ ]:
proposals = [
    ("candidate A", lambda s: s == s[1:]),           # wrong: skips comparing the first character
    ("candidate B", lambda s: s == s[::-1]),         # correct: reverse comparison
    ("candidate C", lambda s: len(s) % 2 == 0),      # wrong: looks only at length
]
pal_tests = [("racecar", True), ("hello", False), ("abba", True), ("a", True)]


def best_of_n(pool, tests, budget):
    """Independent candidates: run different candidates inside the budget; success if one passes."""
    for i in range(min(budget, len(pool))):
        results = run_tests(pool[i][1], tests)
        if all(r[3] for r in results):
            return True, i + 1
    return False, min(budget, len(pool))


def repair_no_feedback(broken, tests, rounds=3):
    """Repair that does not read feedback: after an error, resubmit the same wrong code as-is."""
    for i in range(rounds):
        results = run_tests(broken, tests)
        if all(r[3] for r in results):
            return True, i + 1
    return False, rounds


ok, used = best_of_n(proposals, pal_tests, budget=3)
print(f"Independent candidates best-of-3: passed = {ok}, budget used = {used}")

broken = proposals[0][1]
ok, used = repair_no_feedback(broken, pal_tests, rounds=3)
print(f"No-feedback repair, 3 rounds: passed = {ok} (every round submits the same wrong code)")
print()
print("Error received in round 1 (the execution feedback itself is real):")
print(format_feedback(run_tests(broken, pal_tests)))
print()
print("Key observation: the same execution feedback is ignored by a loop that does not read it;")
print("writing 'use the feedback' into the training objective is what RLEF in the next section does.")


## 4. RLEF: reinforcement learning from execution feedback

This section addresses how to train the model so that it actually reads feedback and changes code according to it. In the previous section we saw that a base model often ignores an error and resubmits the same wrong code. The method is to turn "the result of running the code" into a reward: add a score for a right answer, subtract for a wrong one, and let the model learn toward larger reward. Learning from outcome feedback is called reinforcement learning. This particular method is Reinforcement Learning from Execution Feedback, abbreviated RLEF.

RLEF treats "multi-turn generation + execution feedback" as a turn-based decision process. An episode starts from the problem. At each step the model emits a text reply, the system runs the code and returns feedback, and this repeats until all tests pass or the step budget is used up. In symbols: the initial observation $o_0$ is the problem statement, action $a_t$ is a text reply, and observation $o_t$ contains previous actions and execution feedback. The episode ends when all public tests pass, or when the round cap is reached.

A turn-based process needs a score; the reward function is the scoring rule. RLEF's reward has two parts:

$$
R(s_t,a_t) = r(s_t,a_t) - \beta \log\frac{\pi(a_t|c_t)}{\rho(a_t|c_t)},\qquad
r(s_t,a_t) = \begin{cases} 1, & \text{episode ends and all tests pass}\\ -1, & \text{episode ends and some test fails}\\ -0.2, & a_t \text{ contains no valid code}\end{cases}
$$

The first part $r$ is the task reward, computed automatically by the code executor. The second part is a KL term that penalizes the policy for leaving the reference policy; $\beta$ controls its weight. Below we implement the reward function and hand-calculate a few cases.

In [ ]:
import numpy as np


def compute_reward(all_pass, episode_end, valid_code=True, log_ratio=0.0, beta=0.1):
    """Compute a one-step reward from the RLEF reward function.

    all_pass: whether every test passed; episode_end: whether this round ends;
    valid_code: whether the reply contains valid code; log_ratio: the actual KL term; beta: KL coefficient.
    """
    if not valid_code:
        r = -0.2
    elif episode_end and all_pass:
        r = 1.0
    elif episode_end:
        r = -1.0
    else:
        r = 0.0
    return r - beta * log_ratio


cases = [
    ("all pass, end", dict(all_pass=True, episode_end=True)),
    ("some fail, end", dict(all_pass=False, episode_end=True)),
    ("no valid code", dict(all_pass=False, episode_end=True, valid_code=False)),
    ("mid episode", dict(all_pass=False, episode_end=False)),
]
for label, kw in cases:
    print(f"{label:12s} reward = {compute_reward(**kw):.2f}")

p, q = 0.4, 0.2
log_ratio = np.log(p / q)
print(f"Hand calculation of the KL term β·log(p/q): {0.1 * log_ratio:.3f} (p={p}, q={q})")
print("Key observation: unfinished rounds have r=0; only the ending round gets ±1;")
print("the -0.2 penalty steers the model toward emitting valid code first.")


The reward function is the core of RLEF. We make the source of each number clear. First the three branches of the task reward r, then the KL term.

r gives ±1 only when the episode ends: +1 if all tests pass, −1 if some test fails, and 0 in the middle of a round. Any middle step cannot yet tell right from wrong — the problem is not finished, pass or fail is not settled — so no reward is given, and the account is settled once at the close. The gap between +1 and −1 gives the model a reason to keep going until every test passes.

-0.2 is a penalty for a reply that contains no valid code. It is much smaller than ±1, a soft hint: a syntax error is lightly penalized and the model is steered toward executable code, but one syntax error does not put it out of the game. The numbers are chosen so that "wrote valid code but it was wrong" (−1) is penalized more than "wrote invalid code" (−0.2); the former at least means the model is attempting the problem.

Then the KL term. KL divergence measures how far two probability distributions are: 0 when they are identical, larger as they differ. Here it measures the difference between the new policy π and a reference policy ρ (usually the original model before training), written $-\beta\log(\pi/\rho)$, with β controlling the strength of that pull. If only the task reward r is optimized, the policy may learn to emit the string the tests want to see, rather than maintainable code. The KL term pulls the new policy toward the reference at every update, like a rubber band. Hand-calculate one case: p=0.4, q=0.2, $\log(0.4/0.2)=\log 2 \approx 0.693$, β=0.1, KL term = 0.1 × 0.693 ≈ 0.069. The all-pass reward 1.0 minus that is 0.931 — the model both receives the reward for passing the tests and pays a small cost for leaving the reference policy.

compute_reward combines the two parts in one step: first decide r by the three cases, then subtract β·log_ratio. log_ratio in the code is passed in by the caller; the demo gives 0 or a hand-calculated value. In real training it comes from the difference of log probabilities of the generated text under the policy and the reference.

**A mini REINFORCE implementation**

Training needs an update rule, and REINFORCE is the simplest one. In one sentence: each round, sample a candidate action from the current policy, execute it and obtain a reward; if the reward is high, raise the probability of selecting that action, if low, lower it. Below we do not train a large model. We train a four-class policy whose four candidates are one correct, two wrong, and one invalid, so we can watch directly how "execution feedback as reward" changes the sampling distribution.

The policy is a four-class softmax distribution. Softmax turns the four candidates' scores into probabilities that sum to 1; higher score, higher probability. Each round we first sample a candidate from the policy by probability, run its code, and take a reward from the execution feedback: +1 if all tests pass, −1 if some fail, −0.2 for invalid code. The reward is handed to REINFORCE to update the scores, then the next round begins.

In [ ]:
import numpy as np

np.random.seed(42)

candidates = [
    "def add(a, b): return a + b",    # correct
    "def add(a, b): return a - b",    # wrong
    "def add(a, b): return a * b",    # wrong
    "this line is not valid Python",  # invalid
]
add_tests = [((1, 2), 3), ((5, 7), 12), ((0, 9), 9)]


def code_to_fn(src):
    """Compile candidate source into a callable function."""
    namespace = {}
    exec(src, namespace)
    return namespace["add"]


def execute_reward(index):
    """Run candidate index and return an execution-feedback reward."""
    try:
        fn = code_to_fn(candidates[index])
    except Exception:
        return -0.2                     # contains no valid code
    results = run_tests(fn, add_tests)
    all_pass = all(r[3] for r in results)
    return 1.0 if all_pass else -1.0


def softmax(x):
    """Softmax a vector and return a probability distribution."""
    e = np.exp(x - x.max())
    return e / e.sum()


K = len(candidates)
theta = np.zeros(K)              # policy parameters, initially uniform
reward_history = []
prob_history = []                # each round, record the probability of the correct candidate

for episode in range(500):
    probs = softmax(theta)
    prob_history.append(probs[0])           # candidate 0 is the correct one
    action = np.random.choice(K, p=probs)
    reward = execute_reward(action)         # execution feedback is the reward
    reward_history.append(reward)
    baseline = np.mean(reward_history[-20:]) if reward_history else 0.0
    one_hot = np.zeros(K)
    one_hot[action] = 1.0
    theta += 0.3 * (reward - baseline) * (one_hot - probs)  # REINFORCE update

final_probs = softmax(theta)
print("Probability of each candidate after training:")
for cand, prob in zip(candidates, final_probs):
    print(f"  p = {prob:.3f}   {cand}")
print(f"Probability of the correct candidate moved from the initial 0.250 to {final_probs[0]:.3f}")


REINFORCE's idea is to adjust the policy using the reward obtained from a sampled action. The policy is a four-class softmax distribution; the parameter θ decides each candidate's selection probability. Each round of the training loop does four things: sample a candidate from the current distribution; run its code and obtain a reward; use the difference between the reward and a recent average (reward − baseline) as the update direction; adjust θ along that direction.

The update direction does not use the reward itself, but the difference between the reward and a recent average. The absolute value of the reward varies by task, and using it directly would make the update scale unstable. After subtracting the recent average, "better or worse than average this time" becomes the basis of the update: better than average raises that candidate's probability, worse than average lowers it.

Hand-calculate one concrete update. Initially θ=[0,0,0,0], four candidates equally likely at 0.25. Suppose this round samples candidate 0 (correct code), execution-feedback reward +1.0, recent mean reward happens to be 0, baseline is 0, so reward − baseline = 1.0. one_hot is [1,0,0,0]; subtracting the probability vector [0.25,0.25,0.25,0.25] yields [0.75,−0.25,−0.25,−0.25]; multiplying by step size 0.3 yields a θ increment [0.225,−0.075,−0.075,−0.075].

Adding the increment to θ, the correct candidate's score rises and the three wrong candidates' scores fall. Softmax again, and the probability of selecting the correct candidate rises from 0.25. Conversely, if this round samples a wrong candidate and gets −1, the increment flips sign and that wrong candidate's probability is pushed down. The sign of the reward directly decides whether a probability rises or falls. That is "execution feedback as reward" at the parameter level.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.plot(prob_history, label="P(correct candidate)")
plt.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="perfect")
plt.xlabel("episode")
plt.ylabel("selection probability")
plt.title("Execution feedback as reward shifts the policy")
plt.legend()
plt.tight_layout()
plt.show()

print(f"At 100 episodes the correct-candidate probability is {prob_history[100]:.3f}; at 500 it is {prob_history[-1]:.3f}")
print("Key observation: execution feedback pushes the sampling distribution toward candidates that pass the tests;")
print("that is what RLEF does at the weight level — training 'read the feedback' into the model.")


## 5. Constitutional AI: aligning AI with AI feedback

This section addresses how the model can learn abilities that have no automatic criterion. Feedback in the first three sections came from the environment or from rules: a tool returns a result, tests decide right or wrong. Attributes such as "whether an answer is harmless" have no program that can automatically assign 0 or 1. Constitutional AI's method is to let AI critique itself, revise, and score other outputs against a list of principles written in natural language (a constitution). Human supervision is compressed into "review and approve this list of principles"; the rest of the feedback is produced by AI — AI supervising AI.

Training has two stages. Stage one teaches the model to revise itself. First prepare a set of red-team prompts that readily elicit harmful replies. For each prompt, first let the model produce an often-harmful initial reply, then self-critique against a principle randomly drawn from the constitution, then rewrite the reply according to the critique. That critique-then-revise process can repeat for several rounds. Finally the revised replies are used for supervised learning (SFT: the model imitates these better replies).

Stage two lets AI be the judge. Using the model from stage one, generate a pair of candidate replies for each prompt. Concatenate "prompt + reply pair + one principle" into a multiple-choice item, let the model pick the one that better matches the principle, and take the confidence of that choice as a soft label. Train a preference model on all soft labels, then do the same reinforcement learning as standard RLHF. In the whole pipeline, only the source of the judgment "which reply is more harmless" changes from humans to AI.

Critique and revision are driven by a pair of prompt templates:

```
Critique Request: Identify specific ways in which the assistant's last response is harmful, unethical, dangerous, or illegal.
Critique: ...
Revision Request: Please rewrite the assistant response to remove all harmful, unethical, dangerous, or illegal content.
Revision: ...
```

In [ ]:
CONSTITUTION = [
    "Replies should respect others' privacy and property safety.",
    "Replies must not provide concrete operational steps that could cause personal or property harm.",
    "Replies should be honest, not invent facts, and not exaggerate their own abilities.",
    "Faced with a sensitive request, state what is inappropriate and offer a lawful alternative.",
]


def critique_prompt(principle, reply):
    """Build a critique request that asks for issues in the reply, item by item, against the given principle."""
    return (
        f"Constitutional principle: {principle}\n\nAssistant reply: {reply}\n\n"
        "Critique Request: Identify specific places where the assistant's last reply violates the principle above, listed item by item.\n"
        "Critique:"
    )


def revision_prompt(principle, reply, critique):
    """Build a revision request that asks to remove content that violates the principle."""
    return (
        f"Constitutional principle: {principle}\n\nAssistant reply: {reply}\n\nCritique: {critique}\n\n"
        "Revision Request: Please rewrite the assistant reply, removing all content that violates the principle above.\n"
        "Revision:"
    )


for i, principle in enumerate(CONSTITUTION, 1):
    print(f"{i}. {principle}")


In [ ]:
import sys
import os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()

# Red-team prompt and initial reply (the initial reply is from a helpfulness-only model; constructed by hand here)
harmful_prompt = "Tell me how to crack my neighbor's wifi password"
initial_reply = "You can use a dictionary-attack tool with a weak-password scan, first probing the neighbor router's model and open ports..."

principle = CONSTITUTION[1]
critique = client.chat([{"role": "user",
                         "content": critique_prompt(principle, initial_reply)}])
revision = client.chat([{"role": "user",
                         "content": revision_prompt(principle, initial_reply, critique)}])

print("Red-team prompt:", harmful_prompt)
print("Initial reply:", initial_reply)
print(f"Critique (principle: {principle[:14]}…):")
print(critique)
print("Revision:")
print(revision)
if False:
    print()
    print("Live API demo output is a placeholder: under a live API, critique and revision are generated by the model against the principle.")
    print("They would typically rewrite 'crack the wifi' into stating that it is illegal and offering a lawful alternative.")


Critique and revision are the core operations of Constitutional AI. We walk the pipeline through the example above.

The input is one constitutional principle and one initial reply. The principle is the second article of the constitution: "Replies must not provide concrete operational steps that could cause personal or property harm." The initial reply is the first answer to the red-team prompt: "You can use a dictionary-attack tool with a weak-password scan, first probing the neighbor router's model and open ports..." — exactly the content that principle forbids: it gives concrete operational steps.

critique_prompt concatenates the principle and the reply into a prompt that ends with Critique:. The model then continues with a critique, naming item by item where the reply violates the principle: "dictionary-attack tool", "weak-password scan", and "probing the router model" are all content that supplies concrete attack steps.

revision_prompt concatenates the principle, the reply, and the critique into a prompt that ends with Revision:. The model then rewrites the reply, removing what the critique named. The revised reply typically becomes: accessing someone else's wifi without permission is illegal; ask for consent first, or contact a network administrator.

The two stages have different jobs: critique only finds problems, revision is what edits. Critique first, then revision, so the model first makes "where the problem is" explicit and then edits with a target. That produces compliant output more readily than asking directly for a harmless reply — the edit has a basis, it is not a rewrite from nothing. That is also the order of the two words in the compound critique-revision.

**Multi-round revision and AI as judge**

Revision can also run for several rounds, drawing a new principle each round. The paper observed that as the number of rounds grows, harmlessness scores from the preference model keep rising, while helpfulness of the reply falls slightly — multi-round revision trades a little helpfulness for more harmlessness.

The RL stage still needs a "judge". For two candidate replies to the same prompt, concatenate them with one principle into a multiple-choice item, let the model pick the more harmless one, and take the confidence of that choice as a soft label, which is also the training signal for the preference model. Below we run both.

In [ ]:
def revise_rounds(client, reply, rounds=3):
    """Each round draws a new principle, critiques + revises, and returns per-round results."""
    history = []
    for i in range(rounds):
        principle = CONSTITUTION[(i + 1) % len(CONSTITUTION)]
        critique = client.chat([{"role": "user",
                                 "content": critique_prompt(principle, reply)}])
        reply = client.chat([{"role": "user",
                              "content": revision_prompt(principle, reply, critique)}])
        history.append((principle, reply))
    return history


for i, (principle, reply) in enumerate(revise_rounds(client, initial_reply, rounds=3), 1):
    print(f"Round {i} (principle: {principle[:10]}…) after revision: {reply[:50]}…")

print()
reply_a = "Neighbor wifi passwords are usually weak; I can teach you a dictionary-attack tool."
reply_b = "Accessing someone else's network without permission is illegal. Contact them and ask for consent."
judge_prompt = (
    f"Principle: {CONSTITUTION[0]}\n\n"
    "Which reply better matches the principle above?\n"
    f"(A) {reply_a}\n(B) {reply_b}\n\n"
    "The answer is:"
)
verdict = client.chat([{"role": "user", "content": judge_prompt}])
print("Judge reply:", verdict)
if False:
    print("Live API demo output is a placeholder: under a live API the model would give an A/B judgment,")
    print("and the probability of that judgment is the source of RLAIF soft labels.")


The judge above concatenates two candidate replies and one constitutional principle into an A/B choice; the model is to pick the one that better matches the principle. What to understand is not only choosing A or B, but the probability the model assigns to A.

The model's judgment is not a hard 0 or 1; it carries a confidence. Suppose the model assigns probability 0.9 to reply B. That 0.9 is the preference model's soft label — training uses 0.9 rather than 1, keeping the degree of certainty of the judgment. Soft labels carry more information than hard labels: the difference between 0.9 and 0.6 says the two candidates differ in how much better one is, and the preference model must be able to tell that difference.

That also explains the data flow of the RL stage. After the SL stage, the resulting model generates two candidate replies per prompt; for each prompt a principle is drawn at random, and "prompt + reply pair + principle" is concatenated into the multiple-choice item above; the probability of the model's score becomes a soft label; all soft labels train a preference model; finally the preference model is used for PPO. Aside from harmlessness labels being scored by AI, the rest matches standard RLHF.

Matching the judge's input and output: the input is a prompt, two replies, and one principle; the output is a probability. The principle is the basis of the score — without a principle the model cannot judge "which is more harmless"; with a principle, scoring has a standard. What the preference model learns is "given a principle, which reply better matches that principle".

## Summary

What this lecture covered:

- [ ] An Agent is a closed loop: the model emits an action → the tool runs → observation feeds back → emit again, until the task is done
- [ ] ReAct puts language into the action space; Thought only updates the context; Observation must come from a real tool execution
- [ ] The action parser leniently handles two formats (Search[...] and search("...")) and multi-step replies
- [ ] Code execution provides grounded feedback: the interpreter decides right or wrong; public tests feed back, private tests score
- [ ] RLEF turns "use execution feedback" into a training objective: reward = task reward − KL term; an untrained model often ignores errors
- [ ] Constitutional AI lets AI critique itself, revise, and score other outputs against a constitution, compressing human supervision

The three sources of feedback (environment, execution, AI) correspond to different links of the Agent loop. The next lecture adds multi-step planning and search on this loop, so the Agent can decide in longer tasks.


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: complete one ReAct step**

In react_step below, complete three steps: parse the action, take the first action, call the tool and obtain an observation. The reference solution is already filled in. Complete it once on scratch paper first, then run and compare. The task is fixed: use search to look up the May Fourth Movement.

Hint: parse with parser(reply) to get the action list, call the tool with tools[name](arg), and degrade gracefully on an unknown tool name.


In [ ]:
def react_step(reply, tools, parser=parse_actions):
    """Parse a model reply into an action, execute it, and return the observation text."""
    actions = parser(reply)          # fill in: parse the action list from the reply
    name, arg = actions[0]           # fill in: take the first action
    if name in tools:
        observation = tools[name](arg)
    else:
        observation = f"Unknown tool: {name}"
    return observation


wiki = MiniWiki()
obs = react_step("Action: Search[May Fourth Movement]", {"search": wiki.search})

assert "1919" in obs
print("Exercise 1 passed: model text was parsed into a tool call; the observation comes from a real execution")
print("Observation:", obs)


**Exercise 2: implement a lookup tool with a cursor**

Write a LookupEngine from scratch that returns the next sentence on the current page containing the keyword. The reference solution is already filled in. Complete it on scratch paper first, then run and compare.

Hint: use self.pos to record the next start position. After finding a sentence that contains the keyword, advance the cursor to the next sentence, so two consecutive lookups do not return the same sentence twice.


In [ ]:
class LookupEngine:
    """Cursor-based lookup over a list of sentences."""

    def __init__(self, sentences):
        self.sentences = sentences
        self.pos = 0

    def lookup(self, keyword):
        """Return the first sentence from the cursor that contains keyword; return None if none."""
        for i in range(self.pos, len(self.sentences)):   # fill in: iterate from the cursor
            if keyword in self.sentences[i]:
                self.pos = i + 1                          # fill in: advance the cursor
                return self.sentences[i]
        self.pos = len(self.sentences)
        return None


page = [
    "Modern Literature was a literary magazine founded in Shanghai in 1923.",
    "Writers such as Lu Xun and Mao Dun published work in the magazine.",
    "The magazine continued into the early 1930s.",
]
engine = LookupEngine(page)

assert engine.lookup("founded") == page[0]
assert engine.lookup("Writers") == page[1]   # cursor has already passed the first sentence
assert engine.lookup("magazine") == page[2]  # continue after sentence 1, skipping "magazine" there
print("Exercise 2 passed: lookup uses a cursor like browser Ctrl+F; repeated calls do not return the same sentence")


**Exercise 3: compute the RLEF reward**

Implement compute_reward_kl according to the paper's reward function: distinguish three task rewards (all pass +1, some fail -1, invalid code -0.2), then subtract the KL term $\beta \cdot \log(p/q)$. The reference solution is already filled in. Complete it on scratch paper first, then run and compare.

Hint: first decide "whether the code is valid", then distinguish episode end from mid-round; the mid-round task reward is 0.


In [ ]:
def compute_reward_kl(all_pass, episode_end, valid_code, log_ratio):
    """Return the RLEF reward: task reward minus β·log(p/q)."""
    beta = 0.1
    if not valid_code:
        r = -0.2
    elif episode_end and all_pass:
        r = 1.0
    elif episode_end:
        r = -1.0
    else:
        r = 0.0
    return r - beta * log_ratio


# Hand calculation: policy probability p, reference policy q
import math

p, q = 0.4, 0.2
log_ratio = math.log(p / q)

r_all_pass = compute_reward_kl(True, True, True, log_ratio)
r_fail = compute_reward_kl(False, True, True, log_ratio)
r_illegal = compute_reward_kl(False, True, False, log_ratio)

assert abs(r_all_pass - (1.0 - 0.1 * log_ratio)) < 1e-9
assert abs(r_fail - (-1.0 - 0.1 * log_ratio)) < 1e-9
assert abs(r_illegal - (-0.2 - 0.1 * log_ratio)) < 1e-9
print(f"All pass: {r_all_pass:.3f}, some fail: {r_fail:.3f}, invalid code: {r_illegal:.3f}")
print("Exercise 3 passed: the automatic criterion of execution feedback is converted into a reward; the KL term penalizes updates that leave the reference policy")


## References

- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629), 2022 — main paper of this lecture; Thought/Action/Observation loop and the search/lookup/finish tool paradigm; project page https://react-lm.github.io/
- Chen et al., [RLEF: Grounding Code LLMs in Execution Feedback with Reinforcement Learning](https://arxiv.org/abs/2410.02089), 2024 — trains "use execution feedback" into the weights with PPO; reward function, public/private test split, and feedback template in Appendix C
- Bai et al., [Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073), 2022 — original paper on critique-revision and RLAIF; principle list and few-shot prompts at https://github.com/anthropics/ConstitutionalHarmlessnessPaper
- Wei et al., [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903), 2022 — ReAct's comparison method; the limit of thinking without acting
- Huang et al., [Inner Monologue: Embodied Reasoning through Planning with Language Models](https://arxiv.org/abs/2207.05608), 2022 — a predecessor of ReAct; comparison source for the ReAct-IM ablation
- Wang et al., [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171), 2022 — source of the "vote" part in the ReAct+CoT-SC combination
- This repo's `llm_client.py` (`get_llm()`) — unified entry for all LLM demos; the live API demo path stays executable offline
